In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
# Directories
CODE_DIR = Path(r"C:\Users\willi\.vscode\Github\ml-from-crowd")
DATA_DIR = Path(r"E:\Research_data\Stocktwits\dataset\v1\data\csv")
FIGURES_DIR = Path(r"C:\Users\willi\.vscode\Github\Figures")
MODEL_DATA_DIR = Path(r"C:\Users\willi\.vscode\Github\Data")

# Load trading results
results = pd.read_pickle(MODEL_DATA_DIR / "trading_daily_results.pkl")

portfolios = results['portfolios']
model_cols = ['lr', 'lr_all']

print(f"Available models: {list(portfolios.keys())}")
for col in model_cols:
    p = portfolios[col]
    print(f"\n{col}: {len(p):,} trading days")
    print(f"  Period: {p['date'].min().date()} to {p['date'].max().date()}")
    print(f"  Columns: {list(p.columns)}")
    print(p.head())

In [14]:
results['portfolios']

{'lr':             long_ret  n_long  short_ret  n_short
 date                                            
 2012-01-03      <NA>     NaN       <NA>      NaN
 2012-01-04      <NA>     NaN  -0.001626   4202.0
 2012-01-05      <NA>     NaN   0.006048   4205.0
 2012-01-06      <NA>     NaN   0.000131   4208.0
 2012-01-09      <NA>     NaN   0.005086   4206.0
 ...              ...     ...        ...      ...
 2022-12-23      <NA>     NaN  -0.003999    643.0
 2022-12-27      <NA>     NaN  -0.029723    558.0
 2022-12-28      <NA>     NaN  -0.010583    536.0
 2022-12-29      <NA>     NaN   0.047915    562.0
 2022-12-30      <NA>     NaN   0.018647    585.0
 
 [2768 rows x 4 columns],
 'lr_all':             long_ret  n_long  short_ret  n_short
 date                                            
 2012-01-03      <NA>     NaN       <NA>      NaN
 2012-01-04      <NA>     NaN  -0.001415    424.0
 2012-01-05      <NA>     NaN   0.007323    428.0
 2012-01-06      <NA>     NaN  -0.001772    422.0
 2012-

# Cumulative returns

In [ ]:
# Compute cumulative returns for each model
cum_returns = {}
for col in model_cols:
    p = portfolios[col].copy()
    p = p.set_index('date').sort_index()
    ls_ret = p['long_ret'].fillna(0) - p['short_ret'].fillna(0)
    cum_returns[col] = {
        'long': np.log(1 + p['long_ret'].fillna(0)).cumsum(),
        'short': np.log(1 + p['short_ret'].fillna(0)).cumsum(),
        'long_short': np.log(1 + ls_ret).cumsum(),
    }

# Summary
print("Final Log Cumulative Returns")
print("=" * 50)
for col in model_cols:
    ls_final = cum_returns[col]['long_short'].iloc[-1]
    print(f"{col}:")
    print(f"  Long:       {cum_returns[col]['long'].iloc[-1]:.4f}  ({np.exp(cum_returns[col]['long'].iloc[-1]) - 1:.2%})")
    print(f"  Short:      {cum_returns[col]['short'].iloc[-1]:.4f}  ({np.exp(cum_returns[col]['short'].iloc[-1]) - 1:.2%})")
    print(f"  Long-Short: {ls_final:.4f}  ({np.exp(ls_final) - 1:.2%})")

In [ ]:
# Long-Short cumulative returns
fig, ax = plt.subplots(figsize=(14, 5))
for col in model_cols:
    ax.plot(cum_returns[col]['long_short'].index, cum_returns[col]['long_short'].values, label=col)
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('Date')
ax.set_ylabel('Log Cumulative Return')
ax.set_title('Long-Short Portfolio: Cumulative Returns')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Long and Short legs separately (one subplot per model)
fig, axes = plt.subplots(1, len(model_cols), figsize=(7 * len(model_cols), 5), sharey=True)
if len(model_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, model_cols):
    ax.plot(cum_returns[col]['long'].index, cum_returns[col]['long'].values, label='Long')
    ax.plot(cum_returns[col]['short'].index, cum_returns[col]['short'].values, label='Short')
    ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Date')
    ax.set_ylabel('Log Cumulative Return')
    ax.set_title(f'{col}: Long vs Short')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()